# Orange Documentation Chunking Exploration

This notebook explores `data/all_documentation.txt`, inspects per-file sections, and prototypes cleaning and chunking rules before we lock them into a script.

In [58]:
from pathlib import Path
import re
from dataclasses import dataclass
from typing import List

import pandas as pd

BASE_DIR = Path("..").resolve()
DOC_PATH = BASE_DIR / "data" / "all_documentation.txt"
DOC_PATH

PosixPath('/Users/yun/develop/RAG-orange/data/all_documentation.txt')

In [59]:
@dataclass
class Section:
    path: str
    text: str


MARKER_RE = re.compile(r"^=== (.+) ===$")


def parse_sections(raw_text: str) -> List[Section]:
    sections: List[Section] = []
    current_path: str | None = None
    current_lines: List[str] = []

    for line in raw_text.splitlines():
        m = MARKER_RE.match(line)
        if m:
            # flush previous
            if current_path is not None:
                sections.append(Section(path=current_path, text="\n".join(current_lines).rstrip()))
            current_path = m.group(1).strip()
            current_lines = []
        else:
            current_lines.append(line)

    if current_path is not None:
        sections.append(Section(path=current_path, text="\n".join(current_lines).rstrip()))

    return sections


raw = DOC_PATH.read_text(encoding="utf-8")
sections = parse_sections(raw)
len(sections)

378

In [60]:
df = pd.DataFrame({
    "path": [s.path for s in sections],
    "text": [s.text for s in sections],
    "n_chars": [len(s.text) for s in sections],
})
df

,path,text,n_chars
0,orange3-addons/README.txt,ORANGE ADD-ON LISTS\n\nOrange3 allows installa...,475
1,orange3-addons/OFFICIAL_ADDONS.txt,Orange3-Associate\nOrange3-Bioinformatics\nOra...,327
2,orange3-doc-visual-programming/requirements.txt,"Sphinx>=4.2.0,<8\nsphinx-multiproject\nmyst-pa...",48
3,orange3-doc-visual-programming/source/index.rst,=========================\nOrange Visual Progr...,3823
4,orange3-doc-visual-programming/source/exportin...,# Exporting Visualizations\n\nVisualizations a...,2114
...,...,...,...
373,orange3-timeseries/doc/widgets/time_slice.md,Time Slice\n==========\n\nSelect a slice of me...,1732
374,orange3-timeseries/doc/widgets/difference.md,Difference\n==========\n\nMake the time series...,1175
375,orange3-timeseries/doc/widgets/granger_causali...,Granger Causality\n=================\n\nTest i...,1099
376,orange3-timeseries/doc/widgets/periodogram_w.md,Periodogram\n===========\n\nVisualize time ser...,775


In [61]:
df["top_dir"] = df["path"].str.split("/", n=1).str[0]
df.groupby("top_dir").size().sort_values(ascending=False)

top_dir
orange3-doc-visual-programming    112
orange3                            60
orange3-text                       56
orange-canvas-core                 40
orange3-bioinformatics             36
orange3-timeseries                 21
orange3-single-cell                19
orange-widget-base                 18
orange3-survival-analysis          14
orange3-addons                      2
dtype: int64

In [62]:
IMAGE_LINE_RE = re.compile(r"^\s*!\[.*\]\(.*\)\s*(\{.*\})?\s*$")
WIDTH_ATTR_RE = re.compile(r"\{[^}]*width=[^}]*\}")
MD_LINK_RE = re.compile(r"\[([^\]]+)\]\(([^)]+)\)")
RST_DIRECTIVE_RE = re.compile(r"^\s*\.\.\s+(toctree|automodule|autoclass|autoattribute)\b")
RST_OPTION_RE = re.compile(r"^\s*:[a-zA-Z0-9_-]+:\s*")
EMPH_RE = re.compile(r"(\*{1,2})([^*]+)\1")   # *text* or **text**
UNDERLINE_RE = re.compile(r"^[-=]{3,}\s*$")   # ==== or ---- lines


def clean_text(text: str) -> str:
    lines: list[str] = []
    for line in text.splitlines():
        if IMAGE_LINE_RE.match(line):
            continue
        stripped = line.strip()
        if UNDERLINE_RE.match(stripped):
            # drop pure underline lines (e.g. ====, ----)
            continue
        line = WIDTH_ATTR_RE.sub("", line)
        line = MD_LINK_RE.sub(lambda m: m.group(1), line)
        if RST_DIRECTIVE_RE.match(line):
            continue
        if RST_OPTION_RE.match(line):
            continue
        # remove simple *emphasis* / **bold** markers
        line = EMPH_RE.sub(lambda m: m.group(2), line)
        lines.append(line)

    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


sample_paths = [
    "orange3-doc-visual-programming/source/widgets/unsupervised/PCA.md",
    "orange3-doc-visual-programming/source/widgets/unsupervised/louvainclustering.md",
    "orange3-single-cell/doc/widgets/filter.md",
]

df_samples = df[df["path"].isin(sample_paths)].copy()
df_samples["cleaned"] = df_samples["text"].map(clean_text)
df_samples[["path", "n_chars"]]

,path,n_chars
8,orange3-doc-visual-programming/source/widgets/...,3087
9,orange3-doc-visual-programming/source/widgets/...,2566
346,orange3-single-cell/doc/widgets/filter.md,1466


In [63]:
for _, row in df_samples.iterrows():
    print("== PATH ==", row["path"])
    print("\n--- RAW (first 40 lines) ---")
    print("\n".join(row["text"].splitlines()[:40]))
    print("\n--- CLEANED (first 40 lines) ---")
    print("\n".join(row["cleaned"].splitlines()[:40]))
    print("\n" + "=" * 80 + "\n")

== PATH == orange3-doc-visual-programming/source/widgets/unsupervised/louvainclustering.md

--- RAW (first 40 lines) ---
Louvain Clustering

Groups items using the Louvain clustering algorithm.

**Inputs**

- Data: input dataset

**Outputs**

- Data: dataset with cluster label as a meta attribute
- Graph (with the Network addon): the weighted k-nearest neighbor graph

The widget first converts the input data into a k-nearest neighbor graph. To preserve the notions of distance, the Jaccard index for the number of shared neighbors is used to weight the edges. Finally, a [modularity optimization](https://en.wikipedia.org/wiki/Louvain_Modularity) community detection algorithm is applied to the graph to retrieve clusters of highly interconnected nodes. The widget outputs a new dataset in which the cluster label is used as a meta attribute.

![](images/LouvainClustering.png)

1. Information on the number of clusters found.
2. **Preprocessing**:
   - *Normalize data*: Center to mean and scale

In [64]:
import fnmatch

ALLOWLIST_PATTERNS = [
    # Core widget docs (visual programming)
    "orange3-doc-visual-programming/source/widgets/**/*.md",
    # User guides
    "orange3-doc-visual-programming/source/exporting-*/index.md",
    "orange3-doc-visual-programming/source/report/index.md",
    "orange3-doc-visual-programming/source/learners-as-scorers/index.md",
    "orange3-doc-visual-programming/source/building-workflows/index.md",
    "orange3-doc-visual-programming/source/loading-your-data/index.md",
    # Add-on widget docs
    "orange3-text/doc/widgets/*.md",
    "orange3-bioinformatics/doc/widgets/*.md",
    "orange3-survival-analysis/doc/widgets/*.md",
    "orange3-single-cell/doc/widgets/*.md",
    "orange3-timeseries/doc/widgets/*.md",
    # Core reference & tutorials
    "orange3/Orange/distance/distances.md",
    "orange3/doc/data-mining-library/source/tutorial/*.rst",
]


def is_allowed(path: str) -> bool:
    return any(fnmatch.fnmatch(path, pat) for pat in ALLOWLIST_PATTERNS)


df["allowed"] = df["path"].map(is_allowed)
df["allowed"].value_counts()

allowed
True     190
False    188
Name: count, dtype: int64

In [65]:
# Detailed inspection of allowed vs dropped sections

allowed_df = df[df["allowed"]].copy()
dropped_df = df[~df["allowed"]].copy()

print("Allowed by top_dir (count):")
display(allowed_df.groupby("top_dir").size().sort_values(ascending=False))

print("\nDropped by top_dir (count):")
display(dropped_df.groupby("top_dir").size().sort_values(ascending=False))

print("\nSample KEPT paths per top_dir (first 5 each):")
for top in allowed_df["top_dir"].unique():
    subset = allowed_df[allowed_df["top_dir"] == top]
    print(f"\n[KEPT] top_dir={top}")
    print(subset["path"].head(5).to_string(index=False))

print("\nSample DROPPED paths per top_dir (first 5 each):")
for top in dropped_df["top_dir"].unique():
    subset = dropped_df[dropped_df["top_dir"] == top]
    print(f"\n[DROPPED] top_dir={top}")
    print(subset["path"].head(5).to_string(index=False))

# Borderline: anything under widgets/ that is currently dropped
borderline = df[(~df["allowed"]) & df["path"].str.contains("widgets/")].copy()
if not borderline.empty:
    print("\n\nBorderline cases: paths containing 'widgets/' but currently DROPPED:")
    print(borderline[["top_dir", "path"]].sort_values(["top_dir", "path"]).to_string(index=False))
else:
    print("\n\nNo borderline 'widgets/' paths are being dropped.")

Allowed by top_dir (count):


top_dir
orange3-doc-visual-programming    110
orange3-text                       30
orange3-bioinformatics             15
orange3-timeseries                 15
orange3-single-cell                10
orange3-survival-analysis           6
orange3                             4
dtype: int64


Dropped by top_dir (count):


top_dir
orange3                           56
orange-canvas-core                40
orange3-text                      26
orange3-bioinformatics            21
orange-widget-base                18
orange3-single-cell                9
orange3-survival-analysis          8
orange3-timeseries                 6
orange3-addons                     2
orange3-doc-visual-programming     2
dtype: int64


Sample KEPT paths per top_dir (first 5 each):

[KEPT] top_dir=orange3-doc-visual-programming
orange3-doc-visual-programming/source/exporting...
orange3-doc-visual-programming/source/learners-...
orange3-doc-visual-programming/source/exporting...
orange3-doc-visual-programming/source/report/in...
orange3-doc-visual-programming/source/widgets/u...

[KEPT] top_dir=orange3-survival-analysis
orange3-survival-analysis/doc/widgets/rank-surv...
orange3-survival-analysis/doc/widgets/as-surviv...
orange3-survival-analysis/doc/widgets/kaplan-me...
orange3-survival-analysis/doc/widgets/cox-regre...
orange3-survival-analysis/doc/widgets/stepwise-...

[KEPT] top_dir=orange3-text
     orange3-text/doc/widgets/concordance.md
 orange3-text/doc/widgets/guardian-widget.md
        orange3-text/doc/widgets/ontology.md
orange3-text/doc/widgets/wikipedia-widget.md
  orange3-text/doc/widgets/preprocesstext.md

[KEPT] top_dir=orange3
orange3/doc/data-mining-library/source/tutorial...
orange3/doc/data-mining-l

In [66]:
def slugify_path(path: str) -> str:
    # Very similar to what the script will do: prefix by area + filename
    lower = path.lower()
    prefix = "doc"
    if lower.startswith("orange3-doc-visual-programming/"):
        prefix = "vp"
    elif lower.startswith("orange3-text/"):
        prefix = "text"
    elif lower.startswith("orange3-bioinformatics/"):
        prefix = "bio"
    elif lower.startswith("orange3-survival-analysis/"):
        prefix = "surv"
    elif lower.startswith("orange3-single-cell/"):
        prefix = "single"
    elif lower.startswith("orange3/"):
        prefix = "core"

    name = Path(path).stem
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    return f"{prefix}-{slug}" if slug else prefix


preview = []
for _, row in df[df["allowed"]].head(10).iterrows():
    cleaned = clean_text(row["text"])
    title = cleaned.splitlines()[0] if cleaned else Path(row["path"]).stem
    preview.append({
        "pmid": slugify_path(row["path"]),
        "title": title.strip("# "),
        "abstract": cleaned,
        "source_path": row["path"],
    })

preview

[{'pmid': 'vp-index',
  'title': 'Exporting Visualizations',
  'abstract': "# Exporting Visualizations\n\nVisualizations are an essential part of data science, and analytical reports are incomplete without them. Orange provides a couple of options for saving and modifying visualizations.\n\nAt the bottom of each widget, there is a status bar. Visualization widgets have a Save icon (second from the left) and a Palette icon (fourth from the left). Save icon saves the plot to the computer. Palette icon opens a dialogue for modifying visualizations.\n\n## Saving a plot\n\nVisualizations in Orange can be saved in several formats, namely .png, .svg, .pdf, .pdf from matplotlib and as a matplotlib Python code. A common option is saving in .svg (scalable vector graphic), which you can edit with a vector graphics software such as Inkscape. Ctrl+C (cmd+C) will copy a .png plot, which you can import with ctrl+V (cmd+V) into Word, PowerPoint, or other software tools.\n\nMatplotlib Python code is id

In [67]:
# Prototype chunking logic in the notebook

def tokenize(text: str) -> list[str]:
    # Simple whitespace tokenization; for these docs 1 word ~ 1 token.
    return text.split()


MIN_SECTION_TOKENS = 50


def split_by_headings(cleaned_text: str) -> list[str]:
    """
    Split a long document into sections by approximate headings,
    then merge any tiny sections (< MIN_SECTION_TOKENS) into
    the following section so we don't get degenerate 1-token chunks.
    """
    lines = cleaned_text.splitlines()
    if not lines:
        return []

    heading_idxs: list[int] = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith("#"):
            heading_idxs.append(i)
            continue
        if i == 0 or not lines[i - 1].strip():
            if not stripped.startswith(("-", "*", "+")) and not stripped[:2].isdigit():
                if len(stripped) <= 80:
                    heading_idxs.append(i)

    if not heading_idxs:
        return [cleaned_text]

    if heading_idxs[0] != 0:
        heading_idxs.insert(0, 0)

    raw_sections: list[str] = []
    for start, end in zip(heading_idxs, heading_idxs[1:] + [len(lines)]):
        chunk_lines = lines[start:end]
        section = "\n".join(chunk_lines).strip()
        if section:
            raw_sections.append(section)

    if not raw_sections:
        return [cleaned_text]

    # merge small sections into the next one
    merged: list[str] = []
    buf = ""
    for sec in raw_sections:
        if buf:
            buf = buf + "\n\n" + sec
        else:
            buf = sec
        if len(tokenize(buf)) >= MIN_SECTION_TOKENS:
            merged.append(buf)
            buf = ""
    if buf:
        if merged:
            merged[-1] = merged[-1] + "\n\n" + buf
        else:
            merged.append(buf)

    return merged or [cleaned_text]


def chunk_by_size(tokens: list[str], max_tokens: int, overlap: int) -> list[str]:
    if not tokens:
        return []
    chunks: list[str] = []
    i = 0
    n = len(tokens)
    while i < n:
        end = min(n, i + max_tokens)
        chunk_tokens = tokens[i:end]
        chunks.append(" ".join(chunk_tokens))
        if end >= n:
            break
        i = max(0, end - overlap)
    return chunks


def split_into_chunks(cleaned_text: str) -> list[str]:
    """
    Apply the same rules as the script:

      - if chunk < 300 tokens: keep as-is (no overlap)
      - if 300–600 tokens: split with 30 token overlap
      - if >600 tokens: split by headings first, then apply above per section
    """
    top_tokens = tokenize(cleaned_text)
    n_tokens = len(top_tokens)

    if n_tokens == 0:
        return []

    def chunk_section(text: str) -> list[str]:
        toks = tokenize(text)
        n = len(toks)
        if n == 0:
            return []
        if n < 300:
            return [text]
        if n <= 600:
            return chunk_by_size(toks, max_tokens=300, overlap=30)
        return chunk_by_size(toks, max_tokens=300, overlap=30)

    if n_tokens <= 600:
        return chunk_section(cleaned_text)

    all_chunks: list[str] = []
    for section in split_by_headings(cleaned_text):
        all_chunks.extend(chunk_section(section))
    return all_chunks

In [68]:
# Build chunk-level DataFrame directly in the notebook

allowed_df = df[df["allowed"]].copy()
allowed_df["cleaned"] = allowed_df["text"].map(clean_text)

chunk_rows = []
for _, row in allowed_df.iterrows():
    path = row["path"]
    cleaned = row["cleaned"]
    base_title = cleaned.splitlines()[0] if cleaned else Path(path).stem
    base_title = base_title.strip("# ").strip()
    base_pmid = slugify_path(path)

    chunks = split_into_chunks(cleaned)
    if not chunks:
        continue

    if len(chunks) == 1:
        chunk_rows.append(
            {
                "pmid": base_pmid,
                "title": base_title,
                "abstract": chunks[0],
                "source_path": path,
            }
        )
    else:
        for idx, chunk_text in enumerate(chunks, start=1):
            pmid = f"{base_pmid}-{idx}"
            title = f"{base_title} (part {idx})"
            chunk_rows.append(
                {
                    "pmid": pmid,
                    "title": title,
                    "abstract": chunk_text,
                    "source_path": path,
                }
            )

chunks_df = pd.DataFrame(chunk_rows)
chunks_df["word_count"] = chunks_df["abstract"].str.split().str.len()

print("Total allowed sections:", len(allowed_df))
print("Total chunks (notebook prototype):", len(chunks_df))
print("\nWord count summary for notebook chunks:")
print(chunks_df["word_count"].describe())

chunks_df[["pmid", "title", "word_count"]].head()

Total allowed sections: 190
Total chunks (notebook prototype): 436

Word count summary for notebook chunks:
count    436.000000
mean     175.220183
std       95.941372
min       13.000000
25%       84.500000
50%      163.000000
75%      290.000000
max      300.000000
Name: word_count, dtype: float64


,pmid,title,word_count
0,vp-index,Exporting Visualizations,293
1,vp-index,Learners as Scorers,89
2,vp-index,Exporting Models,114
3,vp-index,Report,187
4,vp-louvainclustering-1,Louvain Clustering (part 1),300


In [70]:
# Print detailed chunking examples for a few docs

example_paths = [
    "orange3-doc-visual-programming/source/widgets/unsupervised/PCA.md",
    "orange3-doc-visual-programming/source/widgets/unsupervised/tsne.md",
    "orange3-timeseries/doc/widgets/line_chart.md",
]

for path in example_paths:
    sect = allowed_df[allowed_df["path"] == path]
    if sect.empty:
        continue
    cleaned = sect.iloc[0]["cleaned"]
    n_tokens = len(tokenize(cleaned))
    print("\n" + "=" * 80)
    print(f"Source path: {path}")
    print(f"Section tokens (before chunking): {n_tokens}")

    ex_chunks = chunks_df[chunks_df["source_path"] == path].copy()
    if ex_chunks.empty:
        print("No chunks found for this path (likely filtered out).")
        continue

    print(f"Number of chunks: {len(ex_chunks)}")
    for i, row in ex_chunks.sort_values("pmid").iterrows():
        text = row["abstract"]
        tokens = tokenize(text)
        preview = " ".join(tokens[:])
        print(f"\n  Chunk pmid: {row['pmid']}")
        print(f"  Title: {row['title']}")
        print(f"  Tokens in chunk: {len(tokens)}")
        print(f"  Preview: {preview}...")


Source path: orange3-doc-visual-programming/source/widgets/unsupervised/PCA.md
Section tokens (before chunking): 356
Number of chunks: 2

  Chunk pmid: vp-pca-1
  Title: PCA (part 1)
  Tokens in chunk: 300
  Preview: PCA PCA linear transformation of input data. Inputs - Data: input dataset Outputs - Transformed Data: PCA transformed data - Data: original data with PCA components as meta variables - Components: Eigenvectors - PCA: PCA to use as Scorer in Rank Principal Component Analysis (PCA) computes the PCA linear transformation of the input data. It outputs either a transformed dataset with weights of individual instances or weights of principal components. 1. Select how many principal components you wish in your output. It is best to choose as few as possible with variance (parameter Explained variance) covered as high as possible. You can also set how much variance you wish to cover with your principal components. 2. You can normalize data to adjust the values to common scale. If